# Improving & Evaluating RAG
**Topics covered:** why measure · building a small test set · hit rate and MRR · chunk size · keyword search · hybrid search · reranking · checking the generated answer

> Notebook 3 built a working RAG pipeline. But "it seems to work" is not good enough. This notebook shows how to **measure** it and then improve it.

---
## PART 1: Why Evaluate?

Without numbers, every change is a guess: "I made the chunks smaller and it feels better."

**Analogy:** a school exam. You don't know if a teaching method works until students take a test with known answers.

RAG has two parts, so two kinds of test:

| Part | Question | Example metric |
|---|---|---|
| **Retrieval** | Did we find the right chunk? | Hit rate, MRR |
| **Generation** | Is the final answer correct and based on the chunks? | Correctness, faithfulness |

Test retrieval first. It is cheap (no LLM) and most problems start there.

---
## PART 2: The Knowledge Base and a Test Set

Same made-up handbook as last time.

In [1]:
handbook = """
## Annual Leave
Every full-time employee gets 24 days of paid annual leave per year. Up to 5 unused days can be carried over to the next year; the rest expire on 31 March. Leave of more than 3 days must be requested in the Pulse portal at least 2 weeks in advance.

## Sick Leave
Employees get 10 days of paid sick leave per year. A doctor's note is needed if you are out for more than 2 days in a row. Sick days do not carry over.

## Parental Leave
The primary caregiver gets 26 weeks of fully paid parental leave. The secondary caregiver gets 4 weeks. Leave can be taken any time in the first year after the child arrives.

## Public Holidays
Nimbus Labs observes 12 public holidays a year. The full list is published on the Pulse portal every January. If a holiday falls on a weekend, no extra day off is given.

## Remote Work
Teams may work remotely up to 3 days a week. Everyone must be reachable during core hours, 11:00 to 15:00 IST. Tuesday is the office day for all teams, so plan in-person meetings for that day.

## Expenses
Submit expense claims within 30 days of spending. Receipts are required for any item over Rs 2,000. While travelling for work, meals are reimbursed up to Rs 1,200 per day. Claims are filed in the Pulse portal.

## Travel
Book flights at least 14 days ahead through the travel desk. Economy class only for flights under 6 hours. Hotels are capped at Rs 7,000 per night in metro cities.

## Equipment
New joiners get a laptop on day one. Laptops are refreshed every 3 years. A one-time home office stipend of Rs 15,000 is available to set up a desk and chair.

## Learning Budget
Each employee has a learning budget of Rs 40,000 per year for courses, books and conferences. Purchases above Rs 10,000 need manager approval first.

## Device Security
If your laptop or phone is lost or stolen, report it to the security team within 24 hours. The team will remotely lock and wipe the device. Never leave a laptop unattended in a car.

## Network and Passwords
Staff connect to the NimbusSecure WiFi network; visitors use NimbusGuest. A VPN is required to reach production systems. Passwords must be changed every 90 days.

## On-Call
Engineers rotate on-call duty weekly. The on-call allowance is Rs 5,000 per week. For a P1 critical incident, the on-call engineer must respond within 15 minutes.

## Performance Reviews
Performance reviews happen twice a year, in April and October. Each review gives a rating from 1 to 5 and a written summary from your manager.

## Onboarding
Every new joiner is assigned a buddy for the first week and follows a 30-60-90 day plan agreed with their manager.

## Health Insurance
Health insurance covers the employee, spouse and up to 2 children with a Rs 5,00,000 cover. Enrolment must be completed within 30 days of joining.

## Referral Bonus
Employees who refer a successful hire receive a Rs 25,000 bonus after the new hire completes 6 months.
"""

print(len(handbook), "characters,", handbook.count("##"), "sections")

2918 characters, 16 sections


A **test set** is a list of questions where we know what a correct chunk looks like. For each question we store a `keyword` that must appear in a correct chunk.

I wrote the questions with different words from the handbook on purpose ("vacation" vs "annual leave"), so they test meaning, not just matching words.

In [2]:
test_set = [
    ("How many vacation days do I get each year?",            "24 days"),
    ("Which day of the week do all teams come to the office?", "Tuesday"),
    ("What is the daily food allowance when I travel?",       "1,200"),
    ("My phone was stolen, what should I do?",                "24 hours"),
    ("How much money can I spend on courses and books?",      "40,000"),
    ("How quickly must I react to a critical outage on call?", "15 minutes"),
    ("When do appraisals happen?",                            "April and October"),
    ("How much time off do new mothers get?",                 "26 weeks"),
    ("Which WiFi should employees use?",                      "NimbusSecure"),
    ("Who helps me during my first week?",                    "buddy"),
]
print(len(test_set), "test questions")

10 test questions


---
## PART 3: The Metrics

For each question we retrieve the top-k chunks and ask: **is a correct chunk among them, and at what rank?**

- **Hit@k:** the share of questions where a correct chunk appears in the top k. (Hit@3 = 0.8 means 8 out of 10 questions found it in the top 3.)
- **MRR (Mean Reciprocal Rank):** rewards finding it *earlier*. Rank 1 scores 1.0, rank 2 scores 0.5, rank 3 scores 0.33, not found scores 0.

Example: correct chunk at rank 1, 3 and not found gives MRR = (1 + 0.33 + 0) / 3 = 0.44.

**Why two metrics?**
- **Hit@k** answers: *did retrieval find it at all?* It ignores position, so rank 1 and rank 3 both count as a hit.
- **MRR** answers: *how high up was it?* It separates a retriever that puts the answer first from one that buries it.

**How to read them together**
- High Hit@k but low MRR: the right chunk is found, but ranked low. Try better embeddings or reranking.
- Low Hit@k: the right chunk isn't found at all. Try better chunking, a larger k, or a different embedding model.

**Notes**
- Both need a test set: questions paired with the chunk that should answer each one.
- These measure **retrieval only**. A good score doesn't guarantee a good final answer, since the LLM can still misuse the chunks.
- Use the same test set before and after each change (chunk size, k, model), so you can tell whether the change helped.


In [3]:
def evaluate(search_fn, chunks, k=3):
    """search_fn(question, k) -> list of chunk texts, best first."""
    hits, reciprocal_ranks, misses = 0, [], []
    for question, keyword in test_set:
        results = search_fn(question, k)
        rank = next((i + 1 for i, text in enumerate(results) if keyword.lower() in text.lower()), None)
        if rank:
            hits += 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append(question)
    n = len(test_set)
    return {"hit@k": hits / n, "mrr": sum(reciprocal_ranks) / n, "misses": misses}

---
## PART 4: Baseline (Embeddings Only)

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def make_dense_search(chunks):
    vectors = model.encode(chunks, normalize_embeddings=True)
    def search(question, k):
        scores = vectors @ model.encode(question, normalize_embeddings=True)
        return [chunks[i] for i in np.argsort(-scores)[:k]]
    return search

section_chunks = [c.strip() for c in handbook.split("\n\n") if c.strip()]
dense = make_dense_search(section_chunks)

for k in (1, 3):
    r = evaluate(dense, section_chunks, k)
    print(f"k={k}:  hit@k={r['hit@k']:.2f}  mrr={r['mrr']:.2f}")
print("missed at k=3:", evaluate(dense, section_chunks, 3)["misses"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

k=1:  hit@k=0.90  mrr=0.90
k=3:  hit@k=1.00  mrr=0.95
missed at k=3: []


This is our **baseline**. Every improvement below is judged against it. A change that doesn't beat the baseline isn't an improvement.

---
## PART 5: Experiment 1, Chunk Size

We have been using one section per chunk. What if chunks were much smaller or larger? We re-cut the handbook at different sizes and measure each.

In [5]:
import re

def chunk_by_size(text, max_chars):
    """Pack whole sentences into chunks of at most max_chars."""
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text.replace("## ", ""))
    chunks, current = [], ""
    for s in sentences:
        if current and len(current) + len(s) + 1 > max_chars:
            chunks.append(current)
            current = s
        else:
            current = f"{current} {s}".strip()
    if current:
        chunks.append(current)
    return chunks

for size in (80, 200, 400, 1000, 3000):
    chunks = chunk_by_size(handbook, size)
    r = evaluate(make_dense_search(chunks), chunks, k=3)
    print(f"size={size:>5}  chunks={len(chunks):>3}  hit@3={r['hit@k']:.2f}  mrr={r['mrr']:.2f}")

size=   80  chunks= 46  hit@3=0.90  mrr=0.85
size=  200  chunks= 18  hit@3=0.90  mrr=0.80
size=  400  chunks=  8  hit@3=0.90  mrr=0.85
size= 1000  chunks=  3  hit@3=1.00  mrr=0.90
size= 3000  chunks=  1  hit@3=1.00  mrr=1.00


**How to read it:**
- Very small chunks can lose the sentence that has the answer or cut it off from its topic.
- Very large chunks put many topics in one vector, so a specific question matches less clearly.
- The best size depends on the data. Here the handbook sections are short, so the natural section chunks are hard to beat. On long, messy documents the gap is usually bigger.

The lesson isn't a magic number. It is: **try sizes and measure.**

**Reading the chunk size results**

- Size 3000 scores a perfect 1.00, but it is only 1 chunk, so the whole document is "retrieved". Nothing is really being searched.
- Size 1000 has 3 chunks and k=3, so every chunk is returned. hit@3 = 1.00 is guaranteed.
- Compare only sizes where chunks is much larger than k (80, 200, 400).
- Among those, **400 is best**: top hit@3 and mrr, with far fewer chunks than 80. Size 80 is too small and can cut a fact in half.
- With only 10 test questions, one question moves a score by 0.10, so small gaps are noise.

**Lesson:** a metric only means something when retrieval is a real choice. If k is close to the number of chunks, a high score tells you nothing.

**Why "retrieval must be a real choice"**

- Think of a quiz: picking 3 correct answers out of 10 options shows skill. Picking 3 out of 3 options always succeeds, so it shows nothing.
- Size 1000: 3 chunks, k=3, so all chunks are returned and the right one is always included. hit@3 = 1.00 is guaranteed.
- Size 80: 46 chunks, k=3, so the retriever must find the right 3 out of 46. hit@3 = 0.90 is a real result.
- Rule: trust hit@k only if the number of chunks is much bigger than k.


---
## PART 6: Experiment 2, Keyword Search

Embeddings match **meaning**. Sometimes you need an **exact word**: a product name, an error code, "NimbusSecure". Keyword search (TF-IDF) scores chunks by how many of the question's words they contain, giving rare words more weight.

**Analogy:** embeddings are a librarian who understands what you mean. Keyword search is Ctrl+F. Each is better at different questions.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

def make_keyword_search(chunks):
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(chunks)
    def search(question, k):
        scores = (matrix @ vectorizer.transform([question]).T).toarray().ravel()
        return [chunks[i] for i in np.argsort(-scores)[:k]]
    return search

keyword = make_keyword_search(section_chunks)
r = evaluate(keyword, section_chunks, 3)
print(f"keyword only:  hit@3={r['hit@k']:.2f}  mrr={r['mrr']:.2f}")
print("missed:", r["misses"])

keyword only:  hit@3=0.80  mrr=0.80
missed: ['What is the daily food allowance when I travel?', 'How much time off do new mothers get?']


Look at which questions each method misses. Keyword search fails when the question uses different words than the handbook ("vacation" vs "leave"). Embeddings fail more on exact names and numbers. They fail on **different** questions, which is why combining them helps.

**Why combine keyword search and embeddings**

- **Keyword search** matches exact words. It misses when the wording differs: "vacation" in the question, "leave" in the handbook.
- **Embeddings** match meaning, so "vacation" finds "leave". They are weaker on exact names and numbers: "NimbusSecure" vs "NimbusGuest", or "24 days" vs "26 weeks", can look alike.
- They fail on **different** questions, so each can cover for the other. Running both and merging the results is called **hybrid search**.

---
## PART 7: Experiment 3, Hybrid Search

Run both searches and merge the rankings with **Reciprocal Rank Fusion (RRF)**: each chunk gets `1 / (60 + rank)` from each list, and the scores are added. A chunk ranked high by *both* wins.

```
dense ranking:    [A, B, C]      keyword ranking:  [B, D, A]
A: 1/61 + 1/63 = 0.032      B: 1/62 + 1/61 = 0.032      -> A and B beat C and D
```

**RRF (Reciprocal Rank Fusion)**  
RRF (Reciprocal Rank Fusion) is a simple way to merge the ranked results from several searches into one final ranking. It's the usual way to combine keyword and embedding results in hybrid search.

- Merges ranked lists from keyword and embedding search into one ranking.
- Scores from the two methods aren't comparable, so RRF uses only the **rank**.
- Each list gives a chunk `1 / (60 + rank)` points. Add them up across lists.
- A chunk ranked high in both lists wins. A chunk found by only one method still stays in the results.


In [7]:
def make_hybrid_search(chunks):
    dense_s, keyword_s = make_dense_search(chunks), make_keyword_search(chunks)
    def search(question, k):
        scores = {}
        for ranking in (dense_s(question, 10), keyword_s(question, 10)):
            for rank, text in enumerate(ranking):
                scores[text] = scores.get(text, 0) + 1 / (60 + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)[:k]
    return search

hybrid = make_hybrid_search(section_chunks)
r = evaluate(hybrid, section_chunks, 3)
print(f"hybrid:  hit@3={r['hit@k']:.2f}  mrr={r['mrr']:.2f}")
print("missed:", r["misses"])

hybrid:  hit@3=1.00  mrr=0.95
missed: []


---
## PART 8: Experiment 4, Reranking

Reranking is a two-step search:

1. Fast search gets a **wide** set of candidates (say the top 8)
2. A slower, more accurate **cross-encoder** reads each *(question, chunk)* pair together and re-scores them. Keep the best few.

**Analogy:** a recruiter shortlists 8 CVs quickly, then a hiring manager interviews those 8 and picks the best 3.

In [8]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def make_reranked_search(first_stage, candidates=8):
    def search(question, k):
        pool = first_stage(question, candidates)
        scores = reranker.predict([(question, text) for text in pool])
        return [pool[i] for i in np.argsort(-scores)[:k]]
    return search

reranked = make_reranked_search(hybrid)
r = evaluate(reranked, section_chunks, 3)
print(f"hybrid + rerank:  hit@3={r['hit@k']:.2f}  mrr={r['mrr']:.2f}")
print("missed:", r["misses"])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

hybrid + rerank:  hit@3=1.00  mrr=1.00
missed: []


---
## PART 9: Compare Everything

In [9]:
methods = {
    "dense (baseline)":    dense,
    "keyword only":        keyword,
    "hybrid (RRF)":        hybrid,
    "hybrid + rerank":     reranked,
}
print(f"{'method':<20} {'hit@1':>6} {'hit@3':>6} {'mrr':>6}")
for name, fn in methods.items():
    r1, r3 = evaluate(fn, section_chunks, 1), evaluate(fn, section_chunks, 3)
    print(f"{name:<20} {r1['hit@k']:>6.2f} {r3['hit@k']:>6.2f} {r3['mrr']:>6.2f}")

method                hit@1  hit@3    mrr
dense (baseline)       0.90   1.00   0.95
keyword only           0.80   0.80   0.80
hybrid (RRF)           0.90   1.00   0.95
hybrid + rerank        1.00   1.00   1.00


**Be careful reading this table.**

- With only 10 questions, one question changes a score by 0.10. Small differences are **noise**, not proof.
- The handbook is tiny and every section is distinct, so even the baseline may score very high. The gaps between methods grow on larger, messier data, where several chunks look similar.
- Reranking and hybrid cost extra time. Use them when the numbers on **your** test set justify it.

A real test set should have 50 to 200 questions taken from real users. Start small like this, then grow it every time you find a failure: add the question that broke, so it can never silently break again.

---
## PART 10: Evaluating the Answer (Generation)

**Why evaluate the answer, not just retrieval**

Part 3 metrics (hit@k, mrr) only check that the right chunk was *found*. The LLM can still get it wrong while writing the answer, so we check the generation separately.

**Three checks**

- **Correctness:** is the final answer right? Example: the LLM says 14 days, but the handbook says 24.
- **Faithfulness:** does every claim come from the retrieved chunks? Example: the LLM adds a rule that isn't in the handbook. This is hallucination, and it can look correct, which makes it easy to miss.
- **Refusal:** does it say "I don't know" when the chunks don't contain the answer? Example: asked about something out of scope, it invents an answer instead.

**Retrieval vs generation**

| Stage | Question | Metric |
|---|---|---|
| Retrieval | Did we find the right chunk? | hit@k, mrr |
| Generation | Did the LLM use it correctly? | correctness, faithfulness, refusal |

A wrong answer can come from either stage, so measure both. If retrieval is good but the answer is bad, the problem is the prompt or the LLM. If retrieval is bad, fix chunking or search first.

**How to check**

- **Cheap automatic check:** confirm the expected keyword (e.g. "24") appears in the answer. This is quick, but it can't catch extra made-up claims.
- **LLM-as-judge:** a second LLM call grades the answer against the retrieved chunks. This handles faithfulness and refusal, which keyword checks can't. The judge can also be wrong, so spot-check its grades by hand.


In [10]:
import os
from dotenv import load_dotenv

load_dotenv()
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"                             # check console.groq.com for the current model list
if HAS_KEY:
    from groq import Groq
    llm = Groq()

def ask_llm(system, user):
    if not HAS_KEY:
        return "[dry run: no GROQ_API_KEY found, so no LLM was called]"
    response = llm.chat.completions.create(
        model=MODEL,
        max_tokens=1000,
        temperature=0,                                        # same question -> same answer, easier to test
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content

SYSTEM = (
    "You answer questions about the Nimbus Labs employee handbook. Use ONLY the context provided. "
    "If the answer is not in the context, reply exactly: I don't know based on the handbook. "
    "Cite the context numbers you used, like [1]."
)

def answer(question, search_fn, k=3):
    chunks = search_fn(question, k)
    context = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(chunks))
    return ask_llm(SYSTEM, f"Context:\n{context}\n\nQuestion: {question}"), chunks

In [11]:
if HAS_KEY:
    correct = 0
    for question, keyword in test_set:
        text, _ = answer(question, reranked)
        ok = keyword.lower() in text.lower()
        correct += ok
        print("OK  " if ok else "MISS", question)
    print(f"\nanswer contains expected keyword: {correct}/{len(test_set)}")
else:
    print("Add GROQ_API_KEY to your .env file to run the answer check.")

OK   How many vacation days do I get each year?
OK   Which day of the week do all teams come to the office?
OK   What is the daily food allowance when I travel?
MISS My phone was stolen, what should I do?
OK   How much money can I spend on courses and books?
MISS How quickly must I react to a critical outage on call?
MISS When do appraisals happen?
MISS How much time off do new mothers get?
OK   Which WiFi should employees use?
OK   Who helps me during my first week?

answer contains expected keyword: 6/10


In [12]:
# Out-of-scope questions: the right behaviour is "I don't know"
if HAS_KEY:
    for q in ["What is the capital of France?", "What is the CEO's salary?"]:
        print(q, "->", answer(q, reranked)[0])
else:
    print("Needs GROQ_API_KEY.")

What is the capital of France? -> I don't know based on the handbook.
What is the CEO's salary? -> I don't know based on the handbook.


---
## PART 11: Where to Look When Something Fails

Use this order. Each step is cheaper than the next.

1. **Is the right chunk in the knowledge base at all?** (If the info isn't indexed, nothing can find it.)
2. **Did retrieval find it?** Print the top-k chunks. Fix chunking, k, or add hybrid search.
3. **Was it ranked high enough?** Add reranking.
4. **Did the prompt use it?** Print the full prompt, check instructions and chunk order.
5. **Did the LLM follow the prompt?** Tighten the instructions or try a stronger model.

Other common improvements you will meet later: **query rewriting** (have the LLM rephrase a vague question before searching), **metadata filters**, and **more context per chunk**.

---
## Key Takeaways

| Concept | Remember |
|---|---|
| **Test set** | Questions with known correct chunks. Grow it every time something fails |
| **Hit@k** | Was the right chunk in the top k? |
| **MRR** | Rewards ranking the right chunk higher |
| **Baseline** | Measure before changing anything |
| **Chunk size** | Try several and measure |
| **Hybrid search** | Meaning (embeddings) plus exact words (keywords), merged with RRF |
| **Reranking** | Fast search for candidates, cross-encoder for the final order |
| **Small test sets** | Differences are noisy, so don't over-trust them |
| **Debug order** | Knowledge base, retrieval, ranking, prompt, LLM |